In [3]:
import pandas as pd
import numpy as np
import warnings
from scipy.interpolate import PchipInterpolator, CubicSpline, Akima1DInterpolator

warnings.filterwarnings('ignore')

train = pd.read_csv("train_dataset.csv")
test = pd.read_csv("test_dataset.csv")

# 1. Establish an absolute safety net from known historical data
global_median = train['implied_volatility'].median()

test['implied_volatility'] = np.nan
df = pd.concat([train, test], ignore_index=True)

df['pred_pchip'] = np.nan
df['pred_spline'] = np.nan
df['pred_akima'] = np.nan

grouped = df.groupby(['parsed_dt', 'option_type', 'expiry_str'])

for _, group in grouped:
    known = group[group['implied_volatility'].notna()].copy()
    unknown = group[group['implied_volatility'].isna()].copy()
    
    if unknown.empty or len(known) < 2: 
        continue
    
    # CRITICAL FIX 1: Drop duplicates to prevent silent math failures
    known = known.drop_duplicates(subset=['moneyness']).sort_values('moneyness')
    
    x_train = known['moneyness'].values
    y_train = known['implied_volatility'].values
    x_test = unknown['moneyness'].values
    
    try:
        # Kernel 1: PCHIP (Handles extrapolation natively)
        pchip = PchipInterpolator(x_train, y_train, extrapolate=True)
        pred_pchip = pchip(x_test)
        df.loc[unknown.index, 'pred_pchip'] = pred_pchip
        
        # Kernel 2: Spline
        if len(known) >= 4:
            spline = CubicSpline(x_train, y_train, bc_type='natural', extrapolate=True)
            df.loc[unknown.index, 'pred_spline'] = spline(x_test)
        else:
            df.loc[unknown.index, 'pred_spline'] = pred_pchip
            
        # Kernel 3: Akima
        if len(known) >= 3:
            akima = Akima1DInterpolator(x_train, y_train)
            akima_preds = akima(x_test)
            
            # CRITICAL FIX 2: Replace Akima's extrapolation NaNs with PCHIP's valid curve
            akima_preds = np.where(np.isnan(akima_preds), pred_pchip, akima_preds)
            df.loc[unknown.index, 'pred_akima'] = akima_preds
        else:
            df.loc[unknown.index, 'pred_akima'] = pred_pchip
            
    except Exception:
        # If the chain's math collapses entirely, gracefully skip to safety net
        continue

# CRITICAL FIX 3: Catch any lingering edge cases before weighting
df['pred_pchip'] = df['pred_pchip'].fillna(global_median)
df['pred_spline'] = df['pred_spline'].fillna(df['pred_pchip'])
df['pred_akima'] = df['pred_akima'].fillna(df['pred_pchip'])

# Apply Trinity Weights
df['final_value'] = (df['pred_spline'] * 0.25) + (df['pred_pchip'] * 0.35) + (df['pred_akima'] * 0.40)

# Export
df.to_csv("filled_dataset.csv", index=False)
print("Engine complete: filled_dataset.csv generated with NO NULLS.")

Engine complete: filled_dataset.csv generated with NO NULLS.
